# *PHASE 8 — V2 ABLATION STUDY*

Compare three feature sets on the labeled quantum subset:
1. Cheminformatics-only (Morgan FP + RDKit 2D)
2. Quantum-only (xTB / DFT descriptors)
3. Fused (cheminformatics + quantum)

The fused model is promoted to V2 ONLY if it improves both

discrimination (PR-AUC) AND calibration (Brier score).

Input:  
Datasets/data/quantum_descriptors.parquet

Datasets/data/features_combined.parquet

Datasets/data/split_manifest.json

# Output: 
experiment/phase8_v2_ablation/ablation_results.json

Datasets/data/model_predictions_v2.parquet (if fused wins)

In [2]:
# ================================================================
# Cell 1 — Imports and Configuration
# ================================================================
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

# ── Project Paths ──────────────────────────────────────────
PROJECT_ROOT = Path(r"G:\research\ECOAI")
DATA_DIR     = PROJECT_ROOT / "Datasets" / "data"
PHASE8_DIR   = PROJECT_ROOT / "experiment" / "phase8_v2_ablation"

# ── Input ──────────────────────────────────────────────────
INPUT_QUANTUM  = DATA_DIR / "quantum_descriptors.parquet"
INPUT_FEATURES = DATA_DIR / "features_combined.parquet"
INPUT_MANIFEST = DATA_DIR / "split_manifest.json"

# ── Output ─────────────────────────────────────────────────
ABLATION_RESULTS = PHASE8_DIR / "ablation_results.json"
OUTPUT_V2_PREDS  = DATA_DIR / "model_predictions_v2.parquet"
RESULTS_FILE     = PROJECT_ROOT / "experiment" / "phase4_model_training" / "training_results.json"

# ── Constants ──────────────────────────────────────────────
RANDOM_SEED = 42
N_FOLDS = 5
np.random.seed(RANDOM_SEED)

# ── Verify ─────────────────────────────────────────────────
assert INPUT_QUANTUM.exists(),  f"❌ {INPUT_QUANTUM} — Run Phase 7 first!"
assert INPUT_FEATURES.exists(), f"❌ {INPUT_FEATURES} — Run Phase 3 first!"
PHASE8_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("  PHASE 8 — V2 ABLATION STUDY")
print("=" * 60)

  PHASE 8 — V2 ABLATION STUDY


In [2]:
# ================================================================
# Cell 2 — Load and Merge Data
# ================================================================

df_quantum = pd.read_parquet(INPUT_QUANTUM)
df_features = pd.read_parquet(INPUT_FEATURES)

# Determine the best classical model from Phase 4
with open(RESULTS_FILE, "r") as f:
    training_results = json.load(f)
best_classical_model = training_results.get("best_classical_model", "LightGBM")

print(f"\n✅ Base model for ablation: {best_classical_model}")

# Identify quantum feature columns
quantum_feature_cols = [
    "homo_ev", "lumo_ev", "homo_lumo_gap_ev", "total_energy_eh",
    "dipole_debye", "partial_charge_mean", "partial_charge_std",
    "partial_charge_max", "partial_charge_min", "mmff_energy",
]
# Only keep columns that exist and have non-null values
quantum_feature_cols = [
    c for c in quantum_feature_cols
    if c in df_quantum.columns and df_quantum[c].notna().any()
]

print(f"\n  Quantum descriptors loaded: {len(df_quantum)} molecules")
print(f"  Available quantum features: {quantum_feature_cols}")

# ── Filter to labeled molecules with converged QC ─────────
df_q_labeled = df_quantum[
    df_quantum["repellent_active"].notna() &
    df_quantum["convergence"].isin(["converged", "na_fallback"])
].copy()

print(f"  Labeled with converged QC: {len(df_q_labeled)}")

# ── Cheminformatics features for this same subset ─────────
meta_cols = [
    "compound_id", "canonical_smiles", "source_dataset",
    "repellent_active", "scaffold_smiles", "fold_id", "qc_status",
]
chem_feature_cols = [c for c in df_features.columns if c not in meta_cols]

df_chem_subset = df_features[
    df_features["compound_id"].isin(df_q_labeled["compound_id"])
].copy()

# Merge cheminformatics and quantum features
df_merged = df_chem_subset.merge(
    df_q_labeled[["compound_id"] + quantum_feature_cols],
    on="compound_id",
    how="inner",
)

y = df_merged["repellent_active"].values.astype(int)
print(f"\n  Merged dataset: {len(df_merged)} molecules")
print(f"  Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"  Cheminformatics features: {len(chem_feature_cols)}")
print(f"  Quantum features: {len(quantum_feature_cols)}")
print(f"  Fused features: {len(chem_feature_cols) + len(quantum_feature_cols)}")




✅ Base model for ablation: XGBoost

  Quantum descriptors loaded: 500 molecules
  Available quantum features: ['homo_ev', 'lumo_ev', 'homo_lumo_gap_ev', 'total_energy_eh', 'dipole_debye', 'partial_charge_mean', 'partial_charge_std', 'partial_charge_max', 'partial_charge_min', 'mmff_energy']
  Labeled with converged QC: 298

  Merged dataset: 298 molecules
  Label distribution: {0: 150, 1: 148}
  Cheminformatics features: 2102
  Quantum features: 10
  Fused features: 2112


In [3]:
# ================================================================
# Cell 3 — Define Feature Sets
# ================================================================

feature_sets = {
    "cheminformatics_only": chem_feature_cols,
    "quantum_only":         quantum_feature_cols,
    "fused":                chem_feature_cols + quantum_feature_cols,
}

for name, cols in feature_sets.items():
    print(f"  {name:25s}: {len(cols)} features")

  cheminformatics_only     : 2102 features
  quantum_only             : 10 features
  fused                    : 2112 features


In [4]:
# ================================================================
# Cell 4 — Scaffold-Aware CV on Quantum Subset
# ================================================================
# Since the quantum subset may not perfectly map to the original
# fold assignments, we create new scaffold-aware splits for this
# specific subset.

from collections import defaultdict

def scaffold_split_subset(df, n_folds=5, seed=42):
    """Quick scaffold split for the quantum subset."""
    rng = np.random.RandomState(seed)
    fold_ids = np.zeros(len(df), dtype=int)

    scaffold_groups = defaultdict(list)
    for i, (_, row) in enumerate(df.iterrows()):
        sc = row.get("scaffold_smiles", "unknown")
        if pd.isna(sc):
            sc = f"unknown_{i}"
        scaffold_groups[sc].append(i)

    groups = list(scaffold_groups.values())
    groups.sort(key=len, reverse=True)
    rng.shuffle(groups)

    fold_sizes = np.zeros(n_folds, dtype=int)
    for group in groups:
        target = int(np.argmin(fold_sizes))
        for idx in group:
            fold_ids[idx] = target
        fold_sizes[target] += len(group)

    return fold_ids


fold_ids = scaffold_split_subset(df_merged, n_folds=N_FOLDS, seed=RANDOM_SEED)
df_merged["ablation_fold_id"] = fold_ids

print(f"\n  Scaffold-aware {N_FOLDS}-fold split for ablation subset:")
for fold in range(N_FOLDS):
    n = (fold_ids == fold).sum()
    print(f"    Fold {fold}: {n} molecules")




  Scaffold-aware 5-fold split for ablation subset:
    Fold 0: 53 molecules
    Fold 1: 75 molecules
    Fold 2: 52 molecules
    Fold 3: 59 molecules
    Fold 4: 59 molecules


In [5]:
# ================================================================
# Cell 5 — Run Ablation Experiment
# We train the model type identified as best in Phase 4 using
# conservative hyperparameters (since the quantum subset is small).

print("\n" + "=" * 60)
print(f"  ABLATION EXPERIMENT ({best_classical_model})")
print("=" * 60)

ablation_results = {}

for fs_name, fs_cols in feature_sets.items():
    print(f"\n  ── {fs_name} ({len(fs_cols)} features) ──")

    X = df_merged[fs_cols].values
    oof_preds = np.zeros(len(y))
    fold_metrics = []

    for fold in range(N_FOLDS):
        train_mask = fold_ids != fold
        val_mask   = fold_ids == fold

        X_train, y_train = X[train_mask], y[train_mask]
        X_val,   y_val   = X[val_mask],   y[val_mask]

        # Impute + scale
        imp = SimpleImputer(strategy="median")
        X_train_imp = imp.fit_transform(X_train)
        X_val_imp   = imp.transform(X_val)

        sc = StandardScaler()
        X_train_std = sc.fit_transform(X_train_imp)
        X_val_std   = sc.transform(X_val_imp)

        # Train
        if best_classical_model == "XGBoost":
            model = xgb.XGBClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=5,
                subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED, verbosity=0, n_jobs=-1,
                eval_metric="logloss", early_stopping_rounds=30
            )
            model.fit(X_train_std, y_train, eval_set=[(X_val_std, y_val)], verbose=False)
        elif best_classical_model == "RandomForest":
            model = RandomForestClassifier(
                n_estimators=300, max_depth=None, min_samples_split=5, min_samples_leaf=2,
                class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
            )
            model.fit(X_train_std, y_train)
        elif best_classical_model == "LightGBM-DART":
            model = lgb.LGBMClassifier(
                boosting_type="dart", n_estimators=400, learning_rate=0.05, max_depth=6, num_leaves=31,
                subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1,
            )
            model.fit(X_train_std, y_train)
        else: # LightGBM fallback
            model = lgb.LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=20,
                subsample=0.8, colsample_bytree=0.8, min_child_samples=5, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1,
            )
            model.fit(X_train_std, y_train, eval_set=[(X_val_std, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])

        y_pred_proba = model.predict_proba(X_val_std)[:, 1]
        oof_preds[val_mask] = y_pred_proba

        y_pred_class = (y_pred_proba >= 0.5).astype(int)
        fold_metrics.append({
            "fold": fold,
            "roc_auc": float(roc_auc_score(y_val, y_pred_proba)),
            "pr_auc":  float(average_precision_score(y_val, y_pred_proba)),
            "mcc":     float(matthews_corrcoef(y_val, y_pred_class)),
            "brier":   float(brier_score_loss(y_val, y_pred_proba)),
        })

    # Overall OOF metrics
    oof_class = (oof_preds >= 0.5).astype(int)
    overall = {
        "roc_auc": float(roc_auc_score(y, oof_preds)),
        "pr_auc":  float(average_precision_score(y, oof_preds)),
        "mcc":     float(matthews_corrcoef(y, oof_class)),
        "brier":   float(brier_score_loss(y, oof_preds)),
    }

    ablation_results[fs_name] = {
        "n_features": len(fs_cols),
        "overall": overall,
        "fold_metrics": fold_metrics,
        "oof_predictions": oof_preds.tolist(),
    }

    print(f"    ROC-AUC: {overall['roc_auc']:.4f}  |  "
          f"PR-AUC: {overall['pr_auc']:.4f}  |  "
          f"MCC: {overall['mcc']:.4f}  |  "
          f"Brier: {overall['brier']:.4f}")


  ABLATION EXPERIMENT (XGBoost)

  ── cheminformatics_only (2102 features) ──
    ROC-AUC: 0.9287  |  PR-AUC: 0.9459  |  MCC: 0.7455  |  Brier: 0.0981

  ── quantum_only (10 features) ──
    ROC-AUC: 0.9151  |  PR-AUC: 0.9308  |  MCC: 0.6529  |  Brier: 0.1159

  ── fused (2112 features) ──
    ROC-AUC: 0.9281  |  PR-AUC: 0.9449  |  MCC: 0.7441  |  Brier: 0.1023


In [6]:
# ================================================================
# Cell 6 — Compare Feature Sets
# ================================================================

print("\n" + "=" * 60)
print("  ABLATION COMPARISON")
print("=" * 60)

comparison_df = pd.DataFrame({
    name: result["overall"]
    for name, result in ablation_results.items()
}).T

comparison_df["n_features"] = [
    ablation_results[name]["n_features"]
    for name in comparison_df.index
]

print(comparison_df.to_string(float_format="{:.4f}".format))


  ABLATION COMPARISON
                      roc_auc  pr_auc    mcc  brier  n_features
cheminformatics_only   0.9287  0.9459 0.7455 0.0981        2102
quantum_only           0.9151  0.9308 0.6529 0.1159          10
fused                  0.9281  0.9449 0.7441 0.1023        2112


In [7]:
# ================================================================
# Cell 7 — Promotion Decision
# ================================================================
# The fused model is promoted ONLY if it improves BOTH:
#   1. Discrimination (PR-AUC) over cheminformatics-only
#   2. Calibration (Brier score) over cheminformatics-only

print("\n" + "=" * 60)
print("  PROMOTION DECISION")
print("=" * 60)

chem_prauc = ablation_results["cheminformatics_only"]["overall"]["pr_auc"]
chem_brier = ablation_results["cheminformatics_only"]["overall"]["brier"]
fused_prauc = ablation_results["fused"]["overall"]["pr_auc"]
fused_brier = ablation_results["fused"]["overall"]["brier"]

prauc_gain = fused_prauc - chem_prauc
brier_gain = chem_brier - fused_brier   # positive = improvement (lower is better)

print(f"  Cheminformatics-only: PR-AUC = {chem_prauc:.4f}, Brier = {chem_brier:.4f}")
print(f"  Fused:                PR-AUC = {fused_prauc:.4f}, Brier = {fused_brier:.4f}")
print(f"\n  PR-AUC gain:  {prauc_gain:+.4f}  {'✅' if prauc_gain > 0 else '❌'}")
print(f"  Brier gain:   {brier_gain:+.4f}  {'✅' if brier_gain > 0 else '❌'}")

promote_fused = (prauc_gain > 0) and (brier_gain > 0)

if promote_fused:
    print(f"\n  🏆 DECISION: PROMOTE fused model to V2!")
    print(f"     Fused features improve BOTH discrimination AND calibration.")
    selected_v2 = "fused"
else:
    print(f"\n  ⏸️  DECISION: STICK WITH V1 (cheminformatics-only).")
    if prauc_gain <= 0:
        print(f"     Fused does NOT improve PR-AUC.")
    if brier_gain <= 0:
        print(f"     Fused does NOT improve Brier score.")
    selected_v2 = "cheminformatics_only"



  PROMOTION DECISION
  Cheminformatics-only: PR-AUC = 0.9459, Brier = 0.0981
  Fused:                PR-AUC = 0.9449, Brier = 0.1023

  PR-AUC gain:  -0.0010  ❌
  Brier gain:   -0.0042  ❌

  ⏸️  DECISION: STICK WITH V1 (cheminformatics-only).
     Fused does NOT improve PR-AUC.
     Fused does NOT improve Brier score.


In [ ]:
# ================================================================
# Cell 8 — Comparison Visualization
# ================================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "sans-serif"],
    "font.size": 10
})

fig, ax = plt.subplots(figsize=(6.5, 4))

metrics = ["roc_auc", "pr_auc", "mcc", "brier"]
titles  = ["ROC-AUC ↑", "PR-AUC ↑", "MCC ↑", "Brier ↓"]
colors  = ["#4A90D9", "#E67E22", "#27AE60"]

# Short labels
short_labels = [
    fs.replace("cheminformatics_only", "Cheminformatics")
      .replace("quantum_only", "Quantum")
      .replace("fused", "Fused")
    for fs in feature_sets.keys()
]

# Keep groups close together
group_spacing = 0.65
x = np.arange(len(metrics)) * group_spacing

# ↓ Slightly reduced bar width + tiny gap between bars
width = 0.16
inner_gap = 0.01

# Collect values
all_values = {
    fs: [ablation_results[fs]["overall"][m] for m in metrics]
    for fs in feature_sets
}

# Plot grouped bars
for i, (fs, color) in enumerate(zip(feature_sets, colors)):
    values = all_values[fs]
    offset = (i - 1) * (width + inner_gap)

    bars = ax.bar(
        x + offset,
        values,
        width=width,
        label=short_labels[i],
        color=color,
        alpha=0.8
    )

    # Annotations
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            val + 0.01,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            color="black",
            rotation=0
        )

# Axes formatting
ax.set_xticks(x)
ax.set_xticklabels(titles, fontsize=10, fontweight="bold", color="black")
ax.set_ylabel("Score", fontsize=10, fontweight="bold", color="black")

# Y-axis ticks bold and black
for label in ax.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")

# Headroom
max_val = max([max(v) for v in all_values.values()])
ax.set_ylim(0, max_val + 0.14)

# Spines and Grid
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
ax.tick_params(axis='y', which='both', length=4, width=1, color='black', left=True)

# Legend
legend = ax.legend(
    frameon=False,
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.10)
)

for text in legend.get_texts():
    text.set_fontweight("bold")
    text.set_color("black")

fig.tight_layout()

# Save
fig.savefig(
    PHASE8_DIR / "xgb_ablation_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()
print(f"\n📊 Saved: xgb_ablation_comparison.png")

📊 Saved: xgb_ablation_comparison.png


In [9]:
# ================================================================
# Cell 9 — Save Ablation Results
# ================================================================

ablation_output = {
    "pipeline":       "Phase 8 — V2 Ablation Study",
    "random_seed":    RANDOM_SEED,
    "n_folds":        N_FOLDS,
    "dataset_size":   len(df_merged),
    "feature_sets": {
        name: {"n_features": len(cols), "features": cols}
        for name, cols in feature_sets.items()
    },
    "results":        {k: {kk: vv for kk, vv in v.items() if kk != "oof_predictions"}
                       for k, v in ablation_results.items()},
    "promotion_decision": {
        "promoted_to_v2":     promote_fused,
        "selected_feature_set": selected_v2,
        "prauc_gain":         float(prauc_gain),
        "brier_gain":         float(brier_gain),
    },
}

# Remove oof_predictions from saved results to keep JSON small
with open(ABLATION_RESULTS, "w") as f:
    json.dump(ablation_output, f, indent=2, default=str)

print(f"✅ Ablation results saved: {ABLATION_RESULTS}")

✅ Ablation results saved: G:\research\ECOAI\experiment\phase8_v2_ablation\ablation_results.json


In [10]:
# ================================================================
# Cell 10 — Final Summary
# ================================================================

print("\n" + "=" * 60)
print("  PHASE 8 COMPLETE — V2 Ablation Study")
print("=" * 60)
print(f"\n  Comparison (on {len(df_merged)} labeled quantum molecules):")
print(comparison_df[["n_features", "pr_auc", "brier"]].to_string(
    float_format="{:.4f}".format))
print(f"\n  Decision: {'🏆 FUSED → V2' if promote_fused else '⏸️ Stay with V1'}")
print(f"\n  Artifacts:")
print(f"    1. {ABLATION_RESULTS}")
print(f"    2. ablation_comparison.png")
print(f"\n  Ready for Phase 9 (Virtual Screening) →")
print("=" * 60)



  PHASE 8 COMPLETE — V2 Ablation Study

  Comparison (on 298 labeled quantum molecules):
                      n_features  pr_auc  brier
cheminformatics_only        2102  0.9459 0.0981
quantum_only                  10  0.9308 0.1159
fused                       2112  0.9449 0.1023

  Decision: ⏸️ Stay with V1

  Artifacts:
    1. G:\research\ECOAI\experiment\phase8_v2_ablation\ablation_results.json
    2. ablation_comparison.png

  Ready for Phase 9 (Virtual Screening) →
